# DistilBERT — ISOT değerlendirmesi
ISOT, WELFake içinde bulunduğu için bu değerlendirme bağımsız dış test değildir. Eğitimde görülen ve görülmeyen haberlerin sonuçları ayrı raporlanır. GPU seçin ve hücreleri sırayla çalıştırın.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DRIVE = Path('/content/drive/MyDrive/fake_news_detection')
PREPARED = PROJECT_DRIVE / 'data/prepared/welfake_temiz_v2_seed42'
MODEL = PROJECT_DRIVE / 'artifacts_v2/deneyler_hizli/04_DistilBERT_256Token_v2_Tekrar/en_iyi_model'
OUTPUT = PROJECT_DRIVE / 'artifacts_v2/ISOT_Test/DistilBERT_256Token'
print('Model:', MODEL)
print('Çıktı:', OUTPUT)

In [ ]:
import subprocess, sys
PROJECT = Path('/content/fake-news-detection')
if (PROJECT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
elif not PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/burak-dursunn/fake-news-detection.git', str(PROJECT)], check=True)
else:
    raise RuntimeError(f'Depo olmayan klasör: {PROJECT}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt'), 'kagglehub'], check=True)

In [ ]:
import kagglehub
dataset_dir = Path(kagglehub.dataset_download('rahulogoel/isot-fake-news-dataset'))
fake = list(dataset_dir.rglob('Fake.csv'))
true = list(dataset_dir.rglob('True.csv'))
if len(fake) != 1 or len(true) != 1:
    raise FileNotFoundError(f'Fake.csv ve True.csv tekil olarak bulunamadı: {dataset_dir}')
print('ISOT dosyaları:', fake[0], true[0])
if not (MODEL / 'config.json').is_file():
    raise FileNotFoundError(f'Eğitilmiş model bulunamadı: {MODEL}')
for file in ['welfake_temiz.csv', 'welfake_split_seed42.csv']:
    if not (PREPARED / file).is_file():
        raise FileNotFoundError(f'v2 verisi bulunamadı: {PREPARED / file}')

In [ ]:
import shutil, torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab çalışma zamanında GPU seçin.')
LOCAL_PREPARED = Path('/content/isot_welfake_v2')
LOCAL_PREPARED.mkdir(exist_ok=True)
for file in ['welfake_temiz.csv', 'welfake_split_seed42.csv']:
    shutil.copy2(PREPARED / file, LOCAL_PREPARED / file)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
command = [sys.executable, '-u', '-m', 'src.evaluate_isot', '--fake', str(fake[0]), '--true', str(true[0]), '--prepared-dir', str(LOCAL_PREPARED), '--model', str(MODEL), '--output', str(OUTPUT), '--max-length', '256']
with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError(f'ISOT testi hata ile durdu: {process.returncode}')